# 02 — Preparazione dei dati, un passaggio alla volta

Obiettivo: conservare i dati originali, definire le 41 feature e l’encoding, costruire il target a
cinque famiglie e uno split ripetibile. **Definire** un encoder non significa apprenderlo:
il primo **fit** avviene soltanto dopo lo split, sulla parte train. Nessun modello viene scelto qui.

In [ ]:
from pathlib import Path
import sys
import json
import hashlib

import numpy as np
import pandas as pd
from IPython.display import display

project_root = Path.cwd() if (Path.cwd() / 'src').is_dir() else Path.cwd().parent
sys.path.insert(0, str(project_root))
from src.data import COLUMN_NAMES, FEATURE_COLUMNS, CATEGORICAL_COLUMNS

## 1. Cleaning: controllare prima di cambiare

Il notebook 01 e [l’audit](../docs/DATA_AUDIT.md) non rilevano mancanti o infiniti.
Conserviamo tutte le righe e tutti i valori; niente imputazione, deduplica o ricampionamento.
Conserviamo `su_attempted=2` perché una ricodifica richiederebbe un significato verificato;
conserviamo `num_outbound_cmds` costante perché non facciamo selezione delle feature.
I controlli seguenti sono condizioni di ingresso: un dato nuovo che li viola richiede una decisione.Leggiamo con `pd.read_csv`, la funzione originale pandas. Dal file locale `src/data.py`
importiamo soltanto le costanti dei nomi delle colonne, non una funzione di caricamento.
`header=None` conserva la prima riga; `names` assegna i nomi perché il CSV non ha intestazione.


In [ ]:
train_path = project_root / 'data/raw/KDDTrain+.csv'
string_types = {column: 'string' for column in (*CATEGORICAL_COLUMNS, 'attack_label')}
train = pd.read_csv(train_path, header=None, names=COLUMN_NAMES, dtype=string_types)
assert train.shape[1] == 43 and len(train) > 0
assert not train.isna().any().any(), 'Valori mancanti: riesaminare il cleaning'
assert np.isfinite(train.select_dtypes(include='number').to_numpy()).all()
print(f'{len(train):,} righe conservate; nessuna modifica ai valori originali.')


## 2. Nomi e ruoli delle feature

Ogni riga descrive una connessione. I nomi e il significato sono quelli della
[specifica UCI](https://kdd.ics.uci.edu/databases/kddcup99/task.html), con legenda nel notebook 01.
Le **3 feature categoriche** sono protocollo (`protocol_type`), servizio (`service`) e stato
della connessione (`flag`). Le altre **38** comprendono misure, contatori, indicatori e quote;
sono numeriche nel file, non necessariamente variabili continue.

`attack_label` è la risposta grezza, `family` sarà il target derivato; `difficulty` è un’annotazione
costruita usando classificatori precedenti, non una proprietà da usare per prevedere la risposta.
**Tutte e tre restano fuori da X.** Non assegniamo numeri ordinati alle categorie.

In [ ]:
X = train.loc[:, list(FEATURE_COLUMNS)].copy()
feature_roles = pd.DataFrame({
    'feature': X.columns,
    'ruolo': ['categoria' if c in CATEGORICAL_COLUMNS else 'numerica' for c in X.columns],
    'trasformazione': ['one-hot' if c in CATEGORICAL_COLUMNS else 'invariata' for c in X.columns],
})
display(feature_roles)
assert X.shape == (len(train), 41)
assert not {'attack_label', 'family', 'difficulty'} & set(X.columns)

## 3. Definizione dell’encoding (T05): ancora nessun fit

`OneHotEncoder(handle_unknown="ignore")` crea un indicatore per categoria; una categoria nuova
in validation produce zeri nel relativo gruppo, senza cambiare il vocabolario.
Le 38 numeriche passano invariate. Non occorre scaling per l’albero: niente scaler, SMOTE o selezione.
La scelta delle colonne è esplicita e le eventuali extra vengono scartate.
I nomi degli indicatori saranno disponibili **dopo il fit**, quando le categorie saranno state
apprese soltanto dal train pertinente. Conserviamo il formato sparso quando conviene.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

numeric_columns = [column for column in FEATURE_COLUMNS if column not in CATEGORICAL_COLUMNS]
preprocessor = ColumnTransformer([
    ('categorical', OneHotEncoder(handle_unknown='ignore'), list(CATEGORICAL_COLUMNS)),
    ('numeric', 'passthrough', numeric_columns),
], remainder='drop')

print('Trasformatore definito; vocabolario non ancora appreso.')
print('Categoriche:', ', '.join(CATEGORICAL_COLUMNS))
print('Numeriche invariate:', len(numeric_columns))


## 4. Labelling del target: attacco → famiglia (T06)

Ordine fisso: **Normal, DoS, Probe, R2L, U2R**. Non è l’encoding delle feature: è la risposta
che vogliamo prevedere. Le 22 tipologie di attacco train seguono la
[mappa UCI](https://kdd.ics.uci.edu/databases/kddcup99/training_attack_types).
Le estensioni e le incertezze test-only sono documentate in [LABEL_MAPPING.md](../docs/LABEL_MAPPING.md).
In particolare `worm→DoS`, `sqlattack→U2R`, `snmp*→R2L` sono convenzioni operative esplicite;
non le attribuiamo a una tabella primaria UNB completa. Non interessano questo split del train.
Una label sconosciuta o mancante causa un errore; nessuna riga viene persa o normalizzata di nascosto.

In [ ]:
from src.targets import FAMILY_ORDER, ATTACK_GROUPS, ATTACK_TO_FAMILY

# ATTACK_TO_FAMILY è la tabella documentata di corrispondenza, non una funzione.
unknown = train['attack_label'].isna() | ~train['attack_label'].isin(ATTACK_TO_FAMILY)
if unknown.any():
    raise ValueError('Etichette train mancanti o non mappate: riesaminare la mappa.')
y = train['attack_label'].map(ATTACK_TO_FAMILY).rename('family')
display(y.value_counts().reindex(FAMILY_ORDER).rename('record training ufficiale'))
display(pd.DataFrame([
    {'famiglia': family, 'etichette grezze': ', '.join(attacks)}
    for family, attacks in ATTACK_GROUPS.items()
]))

# Solo copertura delle etichette: niente fit o metriche del test ufficiale.
test_labels = pd.read_csv(
    project_root / 'data/raw/KDDTest+.csv', header=None, names=COLUMN_NAMES,
    usecols=['attack_label'], dtype={'attack_label': 'string'},
)['attack_label']
assert test_labels.notna().all() and test_labels.isin(ATTACK_TO_FAMILY).all()
print('Copertura completa delle etichette test; righe grezze conservate.')


## 5. Split con scikit-learn: train e validation

Usiamo direttamente [train_test_split](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html):
restituisce `X_train`, `X_validation`, `y_train`, `y_validation` nello stesso ordine.

- `test_size=0.2`: riserviamo il 20% alla validation interna.
- `random_state=42`: possiamo ricreare la stessa suddivisione.
- `stratify=y`: manteniamo circa le proporzioni delle cinque famiglie.

Il nome sklearn è train/test split; qui chiamiamo la seconda parte **validation** perché
proviene da KDDTrain+. KDDTest+ è il test ufficiale separato e non entra nello split.
Prima usavamo `make_validation_split`, una funzione locale che richiamava la stessa API.
La chiamata esplicita mantiene gli stessi indici, senza cambiare il protocollo.

Il CSV conserva un indice 0, 1, ...: le etichette dell'indice coincidono con le posizioni.
Salviamo queste posizioni insieme all'hash del file. Indici disgiunti non garantiscono
feature diverse: misuriamo anche questo limite, senza risuddividere i dati a posteriori.


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_validation, y_train, y_validation = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

# Il loader conserva l'ordine del CSV: qui indice e posizione coincidono.
assert X.index.equals(pd.RangeIndex(len(X)))
train_idx = X_train.index.to_numpy()
validation_idx = X_validation.index.to_numpy()
assert not np.intersect1d(train_idx, validation_idx).size
np.testing.assert_array_equal(np.sort(np.r_[train_idx, validation_idx]), np.arange(len(y)))
assert set(y_train.unique()) == set(y_validation.unique()) == set(FAMILY_ORDER)

display(pd.DataFrame({
    'train': y_train.value_counts(), 'validation': y_validation.value_counts(),
}).reindex(FAMILY_ORDER))


In [ ]:
processed = project_root / 'data/processed'
processed.mkdir(parents=True, exist_ok=True)
np.savez(processed / 'validation_split.npz', train=train_idx, validation=validation_idx)
split_metadata = {
    'source': 'data/raw/KDDTrain+.csv', 'source_sha256': hashlib.sha256(train_path.read_bytes()).hexdigest(),
    'rows': len(train), 'test_size': 0.2, 'random_state': 42,
    'index_kind': 'zero-based positional', 'family_order': list(FAMILY_ORDER),
}
(processed / 'validation_split.json').write_text(json.dumps(split_metadata, indent=2), encoding='utf-8')
print('Indici e metadati salvati in data/processed/ (ignorata da Git).')

In [ ]:
keys = list(FEATURE_COLUMNS)
shared = train.iloc[train_idx].merge(train.iloc[validation_idx], on=keys, suffixes=('_train', '_validation'))
overlap = {
    'vettori_feature_condivisi': len(shared.drop_duplicates(subset=keys)),
    'vettori_con_label_discordanti': len(shared.loc[shared.attack_label_train.ne(shared.attack_label_validation)].drop_duplicates(subset=keys)),
}
display(pd.Series(overlap))
(processed / 'validation_overlap.json').write_text(json.dumps(overlap, indent=2), encoding='utf-8')

## 6. Fit e transform: apprendere soltanto dal train

Ora, e solo ora, `fit_transform(X_train)` apprende le categorie. Sulla validation chiamiamo
soltanto `transform`. Controlliamo forma, valori finiti e categorie eventualmente nuove.
Nel notebook 03 la Pipeline ripeterà correttamente questo fit sullo **stesso train**;
non trasferiamo un encoder adattato all’intero dataset.

In [ ]:
encoded_train = preprocessor.fit_transform(X_train)
encoded_validation = preprocessor.transform(X_validation)
for matrix in (encoded_train, encoded_validation):
    values = matrix.data if hasattr(matrix, 'toarray') else matrix
    assert np.isfinite(values).all()
assert encoded_train.shape[1] == encoded_validation.shape[1]
print('Train trasformato:', encoded_train.shape)
print('Validation trasformata:', encoded_validation.shape)
print('Formato:', type(encoded_train).__name__)
feature_names = preprocessor.get_feature_names_out()
display(pd.Series(feature_names, name='primi 15 nomi dopo encoding').head(15))

In [ ]:
encoder = preprocessor.named_transformers_['categorical']
category_checks = []
for column, learned in zip(CATEGORICAL_COLUMNS, encoder.categories_):
    assert set(learned) == set(X_train[column].unique())
    category_checks.append({
        'feature': column, 'categorie apprese nel train': len(learned),
        'nuove in validation': sorted(set(X_validation[column]) - set(learned)),
    })
display(pd.DataFrame(category_checks))

## Esito e limiti

Gli input sono le sole 41 feature. Cleaning conservativo, encoding, target, split e fit restano
passaggi distinti. Gli indici non si sovrappongono; eventuali feature identiche con label diverse
sono visibili nel controllo sopra. Questa è una validazione casuale stratificata, non una prova
di generalizzazione a nuove reti/tempi. Un eventuale split per gruppi è un esperimento futuro.
Prossimo: baseline e albero fisso, sulle stesse righe di validation.